## One-hot encode new dataframe

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
COLS = pd.read_csv("../data/COLS.csv").set_index('col_name')
MENTION = pd.read_parquet('../data/MENTION.parquet')
print(MENTION.shape)
MENTION.head(3)

In [12]:
from sklearn.preprocessing import OneHotEncoder

cols_short = COLS[COLS.useful == True].index
enc = OneHotEncoder(sparse_output=True, dtype="uint8")
X = enc.fit_transform(MENTION[cols_short])  # returns sparse matrix directly

In [13]:
dummies = []
lovs = []
for i, col in enumerate(COLS.query("useful == 1").index):
    df = pd.get_dummies(MENTION[col], dtype=int)
    lovs.append((col, df.columns.to_list()))
    df.columns = [f"{i}_{j}" for j in range(len(df.columns))]
    dummies.append(df)

In [14]:
M = pd.concat(dummies, axis=1)

In [ ]:
M

,0_0,0_1,0_2,0_3,0_4,0_5,0_6,0_7,0_8,0_9,...,15_8316,15_8317,15_8318,15_8319,15_8320,15_8321,15_8322,15_8323,15_8324,15_8325
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112920,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
112921,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
112922,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
112923,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


## Create column dictionary for M

In [ ]:
LOV = pd.DataFrame(lovs)[1].apply(pd.Series).stack().to_frame('val')
LOV.index.names = ['col_num', 'val_num']
LOV = LOV.join(pd.Series([x[0] for x in lovs], name='col_name').rename_axis("col_num"))
LOV['key'] = LOV.apply(lambda x: f"{x.name[0]}_{x.name[1]}", axis=1)
LOV

val   col_name      key
col_num val_num                                    
0       0                        full_name      0_0
        1               #ERROR!  full_name      0_1
        2                ()via?  full_name      0_2
        3        (Infant) Carey  full_name      0_3
        4               (N)ina?  full_name      0_4
...                         ...        ...      ...
15      8321         FC1880-996  family_id  15_8321
        8322         FC1880-997  family_id  15_8322
        8323         FC1880-998  family_id  15_8323
        8324         FC1880-999  family_id  15_8324
        8325               NULL  family_id  15_8325

[91586 rows x 3 columns]

## Save

In [ ]:
M.to_parquet("../data/M.parquet", index=True)
LOV.to_csv("../data/LOV.csv", index=True)

## Reduce feature space

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca_engine = PCA(n_components=10)
COMPS = pd.DataFrame(pca_engine.fit_transform(X))

In [ ]:
# sns.clustermap(COMPS, col_cluster=None, cmap='Spectral')
# plt.show()

: 